# 07. 성능 개선을 위한 방법론 연구 (사후 탐색, 개발군 전용)

연구기획서 4.2절(10/4 추가)을 실행한다. 분석 1의 주 결과(9/28, 평가군 1회 평가)는 고정하고, **개발군 8,000명 안에서만** 아래 가설을 같은 중첩 교차검증(외부 5분할 × 2회, 내부 3분할, 탐색 40회, 같은 시드)으로 검정한다. 비교 기준은 주 모델 M1 Elastic-net의 개발군 OOF 예측(노트북 03)이다.

| 번호 | 가설 | 주 비교(Holm 가족) | 보조 |
| --- | --- | --- | --- |
| H1 | 연속 해마 z를 회귀한 뒤 순위를 쓰면 판별력이 오른다 | Elastic-net 회귀 | CatBoost 회귀 |
| H2 | 확장 임상 변수가 신호를 더한다 | M1+ Elastic-net | M1+ CatBoost |
| H3 | 모델 결합이 낫다 | 로지스틱 스태킹 | 로짓 평균 |
| H4 | TabPFN 설정 변경이 도움이 된다 | TabPFN-3.5 앙상블 32 | TabPFN 회귀 |
| H5 | 세 뇌 체적 z의 공통 요인(고정 가중 복합 z)이 잡음을 줄인다 | Elastic-net 복합 z 회귀 | 없음 |
| H6 | (해석용) 학습 곡선으로 성능 상한 확인 | 없음 | 없음 |
| H7 | (보조) 결과 정의를 바꿔도 모델 순위가 유지된다 | 없음 | 없음 |

**판정 규칙(사전 고정)** 주 비교 5건에 Holm 보정. 개발군에서 ΔAUPRC ≥ 0.01이고 Holm p < 0.05인 가설이 있을 때만 그중 가장 단순한 하나를 평가군에서 한 번 더 평가하고 "사후 탐색의 확인 검정(평가군 2차 사용)"으로 표시한다. 이 노트북은 **평가군을 읽지 않는다**. 판정 결과만 출력한다.

실행 스위치: `FAST`(점검용 축소), `RUN_H4`(TabPFN, 별도 프로세스, 약 20분), `RUN_H7`(결과 정의 민감도, 약 30분).

In [1]:
FAST = False
RUN_H4, RUN_H7 = True, True
TABPFN_DEVICE = "auto"      # 노트북 03 0절 점검을 통과한 장치

import sys, json, time, warnings, pickle
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import logging; logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
import matplotlib.font_manager as fm
warnings.filterwarnings("ignore")
_avail = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"] if f in _avail] + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
COL = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "violet": "#4a3aa7", "gray": "#9aa3ad", "ink": "#0b0b0b", "ink2": "#52514e"}

ROOT = Path.cwd()
while not (ROOT / "02_Data").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code"))
import tabular_model_comparison as tmc
import worker_util as wu
import os
_tok = ROOT / "code" / ".tabpfn_token"
if not os.environ.get("TABPFN_TOKEN") and _tok.exists(): os.environ["TABPFN_TOKEN"] = _tok.read_text(encoding="utf-8").strip()

OUT = ROOT / "05_분석결과"; DERIVED, M, TAB, FIG = OUT / "derived", OUT / "models", OUT / "tables", OUT / "figures"
SEED = 20260915; NBOOT = 300 if FAST else 2000; PFX = "fast_" if FAST else ""; prefix = "fast_" if FAST else "full_"
B = pd.read_csv(DERIVED / "base_analysis.csv"); info = json.load(open(DERIVED / "normative_model.json", encoding="utf-8"))
dev = B[B.split == "dev"].reset_index(drop=True)
assert "eval" not in set(dev.split), "이 노트북은 평가군을 쓰지 않는다"
cfg_m1 = json.load(open(M / f"{prefix}M1" / "config.json", encoding="utf-8")); M1 = cfg_m1["features"]
y = dev.atrophy_pos.values.astype(int); ids = dev.T_ID.astype(str).tolist()

def make_cfg(task="classification", models=("enet", "cat")):
    c = tmc.Config(task=task, models=list(models), tabpfn_device=TABPFN_DEVICE)
    if FAST: c.outer_folds, c.repeats, c.inner_folds, c.n_iter, c.n_boot = 3, 1, 2, 4, 200
    return c
def oof_mean(out_dir, model):
    d = pd.read_csv(Path(out_dir) / "oof_predictions.csv"); assert d.iloc[:, 0].astype(str).tolist() == ids, "대상자 순서 불일치"
    cols = [c for c in d.columns if c.startswith(model + "_rep")]; return d[cols].mean(axis=1).values if cols else None
def sens20(yy, s, frac=0.20):
    n = int(round(frac * len(yy))); return yy[np.argsort(-s, kind="stable")[:n]].sum() / yy.sum()
METR = {"AUROC": roc_auc_score, "AUPRC": average_precision_score, "민감도@20%": sens20}
BASE = oof_mean(M / f"{prefix}M1", "enet"); assert BASE is not None
rng = np.random.default_rng(SEED); IDX = [i for i in (rng.integers(0, len(y), len(y)) for _ in range(NBOOT)) if 0 < y[i].sum() < len(i)]
RESULTS = []
def evaluate(name, score, hyp, primary, note=""):
    """score: 클수록 위축 위험. 기준 M1 Elastic-net OOF와 같은 대상자 쌍체 부트스트랩."""
    r = {"가설": hyp, "변형": name, "주 비교": primary, "비고": note}
    for k, f in METR.items(): r[k] = f(y, score)
    for k, f in METR.items():
        d = np.array([f(y[i], score[i]) - f(y[i], BASE[i]) for i in IDX])
        r[f"Δ{k}"] = f(y, score) - f(y, BASE); r[f"Δ{k} 하한"], r[f"Δ{k} 상한"] = np.percentile(d, [2.5, 97.5])
        r[f"Δ{k} p"] = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / (len(d) + 1))
    RESULTS.append(r)
    print(f"  [{hyp}] {name:<34} AUROC {r['AUROC']:.4f}  AUPRC {r['AUPRC']:.4f}  ΔAUPRC {r['ΔAUPRC']:+.4f} ({r['ΔAUPRC 하한']:+.4f}, {r['ΔAUPRC 상한']:+.4f}) p={r['ΔAUPRC p']:.3f}")
print(f"개발군 {len(dev)}명, 양성 {y.sum()}명 | M1 변수 {len(M1)}개 | 기준 M1 Elastic-net OOF: AUROC {roc_auc_score(y, BASE):.4f}, AUPRC {average_precision_score(y, BASE):.4f} | 부트스트랩 {len(IDX)}회")

개발군 8000명, 양성 1214명 | M1 변수 17개 | 기준 M1 Elastic-net OOF: AUROC 0.6306, AUPRC 0.2394 | 부트스트랩 2000회


## H1. 연속 해마 z 회귀 후 순위화

In [2]:
t0 = time.time(); d1 = M / f"method_{PFX}H1_reg_M1"
tmc.run(dev, "T_ID", "z_hip", M1, [], None, make_cfg("regression", ["enet", "cat"]), str(d1), resume=True, compute_only=["enet", "cat"])
print(f"\nH1 실행 {time.time() - t0:.0f}초")
for m, lab, prim in [("enet", "Elastic-net 회귀 → 순위", True), ("cat", "CatBoost 회귀 → 순위", False)]:
    p = oof_mean(d1, m)
    if p is not None: evaluate(lab, -p, "H1", prim)

[설정] task=regression  n=8000  p=17  models=['enet', 'cat']  outer=5x2  inner=3  n_iter=40
  Elastic-net (기준선)      repeat 0 fold 0  done (     2s)
  Elastic-net (기준선)      repeat 0 fold 1  done (     2s)
  Elastic-net (기준선)      repeat 0 fold 2  done (     2s)
  Elastic-net (기준선)      repeat 0 fold 3  done (     2s)
  Elastic-net (기준선)      repeat 0 fold 4  done (     3s)
  Elastic-net (기준선)      repeat 1 fold 0  done (     3s)
  Elastic-net (기준선)      repeat 1 fold 1  done (     3s)
  Elastic-net (기준선)      repeat 1 fold 2  done (     3s)
  Elastic-net (기준선)      repeat 1 fold 3  done (     3s)
  Elastic-net (기준선)      repeat 1 fold 4  done (     3s)
  CatBoost               repeat 0 fold 0  done (     8s)
  CatBoost               repeat 0 fold 1  done (    15s)
  CatBoost               repeat 0 fold 2  done (    22s)
  CatBoost               repeat 0 fold 3  done (    29s)
  CatBoost               repeat 0 fold 4  done (    37s)
  CatBoost               repeat 1 fold 0  done (    44s

## H2. 확장 임상 블록 M1+

기존 블록에 없던 변수를 원자료에서 만든다: 질환 이환 기간(나이 − 진단 나이, 비환자 0, 환자인데 진단 나이 결측이면 0과 지시자), 맥박, 엉덩이둘레, 허리 대 엉덩이 비, 신장, 체중, 간접흡연, 결혼 상태(범주), 소득, 검진 연도, 음주 기간 범주, 폐경 여부와 초경 및 폐경 나이, 임신 경험. 특수 코드 처리는 노트북 01과 같다.

In [3]:
COHORT = ROOT / "02_Data" / "cohort"
def read_clin(fn):
    df = pd.read_csv(COHORT / fn)
    for c in df.columns:
        if c.endswith("_ID") or c.endswith("DATA_CLASS"): continue
        s = pd.to_numeric(df[c], errors="coerce"); s = s.mask(s.isin([66666, 99999])); s = s.mask(s == 77777, np.nan); df[c] = s
    return df
raw = read_clin("BASE_DATA1.txt").merge(read_clin("BASE_DATA2.txt").drop(columns=["T_DATA_CLASS"]), on="T_ID").merge(read_clin("BASE_DATA3.txt").drop(columns=["T_DATA_CLASS"]), on="T_ID")
yes = lambda s: s.map({1: 0, 2: 1})
ext = pd.DataFrame({"T_ID": raw.T_ID})
for dz, ag in [("htn", "T_HTNAG"), ("dm", "T_DMAG"), ("lip", "T_LIPAG")]:
    has = yes(raw[f"T_{dz.upper()}"]) == 1; dur = (raw.T_AGE - raw[ag]).clip(lower=0)
    ext[f"{dz}_dur"] = np.where(has, dur.fillna(0), 0.0); ext[f"{dz}_dur_na"] = (has & raw[ag].isna()).astype(int)
ext["pulse"], ext["hip"], ext["whr"], ext["height"], ext["weight"] = raw.T_PULSE, raw.T_HIP, raw.T_WAIST / raw.T_HIP, raw.T_HEIGHT, raw.T_WEIGHT
ext["psm"], ext["exam_year"], ext["drdu"], ext["marry"] = yes(raw.T_PSM), raw.T_EDATE // 100, raw.T_DRDU.fillna(0), raw.T_MARRY   # 소득(income)은 base_analysis에 이미 있음
male = raw.T_SEX == 1
ext["pmyn"] = np.where(male, 0, yes(raw.T_PMYN)); ext["mnsag"] = raw.T_MNSAG.where(~male); ext["pmag"] = raw.T_PMAG.where(~male); ext["preg"] = np.where(male, 0, yes(raw.T_PREG))
ext.to_csv(DERIVED / "extended_clinical.csv", index=False)
EXT = [c for c in ext.columns if c != "T_ID"] + ["income"]; CAT = ["marry"]
dev2 = dev.merge(ext, on="T_ID", how="left"); assert dev2.T_ID.astype(str).tolist() == ids
print("확장 변수", len(EXT), "개:", EXT); print("결측률(%):", (dev2[EXT].isna().mean() * 100).round(1).to_dict())
t0 = time.time(); d2 = M / f"method_{PFX}H2_M1plus"
tmc.run(dev2, "T_ID", "atrophy_pos", M1 + EXT, CAT, None, make_cfg("classification", ["enet", "cat"]), str(d2), resume=True, compute_only=["enet", "cat"])
print(f"\nH2 실행 {time.time() - t0:.0f}초")
for m, lab, prim in [("enet", "M1+ Elastic-net", True), ("cat", "M1+ CatBoost", False)]:
    p = oof_mean(d2, m)
    if p is not None: evaluate(lab, p, "H2", prim)
# 해석용: 확장 변수 각각의 연령, 성별 보정 해마 z 연관(개발군)
import statsmodels.api as sm
rows = []
WOMEN_ONLY = ["pmyn", "mnsag", "pmag", "preg"]
for c in [e for e in EXT if e not in CAT]:
    dd = dev2[[c, "z_hip", "age", "male"]].dropna()
    if c in WOMEN_ONLY: dd = dd[dd.male == 0]; X = sm.add_constant(dd[[c, "age"]].values.astype(float))
    else: X = sm.add_constant(dd[[c, "age", "male"]].values.astype(float))
    if dd[c].std() == 0: continue
    r = sm.OLS(dd.z_hip.values, X).fit()
    rows.append({"변수": c + (" (여성만)" if c in WOMEN_ONLY else ""), "β(1 SD당)": r.params[1] * dd[c].std(), "p": r.pvalues[1], "n": len(dd)})
U = pd.DataFrame(rows).sort_values("p"); U.to_csv(TAB / f"{PFX}table7b_extended_univariate.csv", index=False, encoding="utf-8-sig")
print("\n확장 변수와 해마 z의 연관(연령, 성별 보정, 해석용):"); print(U.head(8).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

확장 변수 20 개: ['htn_dur', 'htn_dur_na', 'dm_dur', 'dm_dur_na', 'lip_dur', 'lip_dur_na', 'pulse', 'hip', 'whr', 'height', 'weight', 'psm', 'exam_year', 'drdu', 'marry', 'pmyn', 'mnsag', 'pmag', 'preg', 'income']
결측률(%): {'htn_dur': 0.0, 'htn_dur_na': 0.0, 'dm_dur': 0.0, 'dm_dur_na': 0.0, 'lip_dur': 0.0, 'lip_dur_na': 0.0, 'pulse': 4.6, 'hip': 1.5, 'whr': 1.5, 'height': 0.5, 'weight': 0.5, 'psm': 1.5, 'exam_year': 0.0, 'drdu': 0.0, 'marry': 0.7, 'pmyn': 0.8, 'mnsag': 37.1, 'pmag': 62.7, 'preg': 0.4, 'income': 19.8}
[설정] task=classification  n=8000  p=37  models=['enet', 'cat']  outer=5x2  inner=3  n_iter=40
  Elastic-net (기준선)      repeat 0 fold 0  done (    25s)
  Elastic-net (기준선)      repeat 0 fold 1  done (    50s)
  Elastic-net (기준선)      repeat 0 fold 2  done (    79s)
  Elastic-net (기준선)      repeat 0 fold 3  done (   105s)
  Elastic-net (기준선)      repeat 0 fold 4  done (   129s)
  Elastic-net (기준선)      repeat 1 fold 0  done (   158s)
  Elastic-net (기준선)      repeat 1 fold 1  done 

## H3. 모델 결합 (스태킹)

In [4]:
o = pd.read_csv(M / f"{prefix}M1" / "oof_predictions.csv"); assert o.T_ID.astype(str).tolist() == ids
models = [m for m in ["enet", "rf", "lgbm", "xgb", "cat", "tabpfn"] if f"{m}_rep0" in o.columns]
reps = sorted({int(c.rsplit("_rep", 1)[1]) for c in o.columns if "_rep" in c})
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
avg, meta = np.zeros(len(y)), np.zeros(len(y))
for r in reps:
    P = np.column_stack([logit(o[f"{m}_rep{r}"].values) for m in models])
    avg += expit(P.mean(axis=1)) / len(reps)
    meta += cross_val_predict(LogisticRegression(C=1.0, max_iter=2000), P, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED + r), method="predict_proba")[:, 1] / len(reps)
print("결합 대상 모델:", models, "| 반복", reps)
evaluate("로지스틱 스태킹(교차검증 메타학습)", meta, "H3", True, "2차 누출 가능성: 기저 모델이 다른 폴드의 표적을 봄")
evaluate("로짓 단순 평균", avg, "H3", False)

결합 대상 모델: ['enet', 'rf', 'lgbm', 'xgb', 'cat', 'tabpfn'] | 반복 [0, 1]
  [H3] 로지스틱 스태킹(교차검증 메타학습)                AUROC 0.6274  AUPRC 0.2376  ΔAUPRC -0.0018 (-0.0044, +0.0008) p=0.177
  [H3] 로짓 단순 평균                           AUROC 0.6276  AUPRC 0.2345  ΔAUPRC -0.0049 (-0.0117, +0.0027) p=0.193


## H4. TabPFN 설정 (별도 프로세스)

In [5]:
if RUN_H4:
    for tag, task, target, nest, lab, prim, sign in [("tabpfn32", "classification", "atrophy_pos", 32, "TabPFN-3.5 앙상블 32", True, 1),
                                                     ("tabpfn_reg", "regression", "z_hip", 8, "TabPFN-3.5 회귀 → 순위", False, -1)]:
        d4 = M / f"method_{PFX}H4_{tag}_M1"; t0 = time.time()
        spec = {"root": str(ROOT), "block": f"M1 {lab}", "features": M1, "models": ["tabpfn"], "fast": FAST, "device": TABPFN_DEVICE,
                "tabpfn_version": tmc.Config().tabpfn_version, "out_dir": str(d4), "task": task, "target": target, "tabpfn_n_estimators": nest}
        rc, out = wu.run_spec("run", spec)
        print(f"{lab}: {'완료' if rc == 0 else '실패 ' + wu.explain(rc, out)} ({time.time() - t0:.0f}초)")
        if rc == 0 and (d4 / "oof_predictions.csv").exists():
            p = oof_mean(d4, "tabpfn")
            if p is not None: evaluate(lab, sign * p, "H4", prim)
    base_tab = oof_mean(M / f"{prefix}M1", "tabpfn")
    if base_tab is not None: evaluate("TabPFN-3.5 앙상블 8(노트북 03, 참고)", base_tab, "H4", False, "참고")
else:
    print("RUN_H4 = False: 건너뜀")

[단계] M1 TabPFN-3.5 앙상블 32: TabPFN 계산 (장치 auto, classification, 목표 atrophy_pos, 앙상블 32), 나머지 모델은 저장된 결과 재사용
[설정] task=classification  n=8000  p=17  models=['tabpfn']  outer=5x2  inner=3  n_iter=40
  TabPFN-3.5             repeat 0 fold 0  done (    76s)
  TabPFN-3.5             repeat 0 fold 1  done (   151s)
  TabPFN-3.5             repeat 0 fold 2  done (   233s)
  TabPFN-3.5             repeat 0 fold 3  done (   321s)
  TabPFN-3.5             repeat 0 fold 4  done (   405s)
  TabPFN-3.5             repeat 1 fold 0  done (   489s)
  TabPFN-3.5             repeat 1 fold 1  done (   573s)
  TabPFN-3.5             repeat 1 fold 2  done (   657s)
  TabPFN-3.5             repeat 1 fold 3  done (   741s)
  TabPFN-3.5             repeat 1 fold 4  done (   826s)

# 모델 비교 요약 (classification, n=8000, p=17)

외부 5분할 × 2회, 내부 3분할, 모델당 무작위 탐색 40회.
사전 고정 선택 규칙에 따른 **주 모델: TabPFN-3.5** (지표 AUPRC, 허용 오차 0.01).

| label      |   AUPRC_mean |   AUPRC_sd |   AUROC_mean |   Brier_mean |   cal_slope_mean |

## H5. 세 뇌 체적 z의 고정 가중 복합 z 회귀

In [6]:
dev["z_comp"] = (dev.z_hip + dev.z_wb - dev.z_vent) / 3        # 고정 가중(추정 없음)이라 폴드 간 누출이 없다
print("복합 z와 해마 z 상관: %.3f" % np.corrcoef(dev.z_comp, dev.z_hip)[0, 1])
t0 = time.time(); d5 = M / f"method_{PFX}H5_composite_M1"
tmc.run(dev, "T_ID", "z_comp", M1, [], None, make_cfg("regression", ["enet"]), str(d5), resume=True, compute_only=["enet"])
print(f"\nH5 실행 {time.time() - t0:.0f}초")
p = oof_mean(d5, "enet"); evaluate("Elastic-net 복합 z 회귀 → 순위", -p, "H5", True)

복합 z와 해마 z 상관: 0.785
[설정] task=regression  n=8000  p=17  models=['enet']  outer=5x2  inner=3  n_iter=40
  Elastic-net (기준선)      repeat 0 fold 0  done (     1s)
  Elastic-net (기준선)      repeat 0 fold 1  done (     2s)
  Elastic-net (기준선)      repeat 0 fold 2  done (     2s)
  Elastic-net (기준선)      repeat 0 fold 3  done (     2s)
  Elastic-net (기준선)      repeat 0 fold 4  done (     2s)
  Elastic-net (기준선)      repeat 1 fold 0  done (     2s)
  Elastic-net (기준선)      repeat 1 fold 1  done (     3s)
  Elastic-net (기준선)      repeat 1 fold 2  done (     3s)
  Elastic-net (기준선)      repeat 1 fold 3  done (     3s)
  Elastic-net (기준선)      repeat 1 fold 4  done (     3s)

# 모델 비교 요약 (regression, n=8000, p=17)

외부 5분할 × 2회, 내부 3분할, 모델당 무작위 탐색 40회.
사전 고정 선택 규칙에 따른 **주 모델: Elastic-net (기준선)** (지표 MAE, 허용 오차 0.01).

| label                |   MAE_mean |   MAE_sd |   RMSE_mean |   R2_mean |   train_seconds |
|:---------------------|-----------:|---------:|------------:|----------:|---------------

## H6. 학습 곡선 (성능 상한 해석)

In [7]:
fin = pickle.load(open(OUT / "final_models" / "M1__enet.pkl", "rb")); bp = fin["best_params"]
print("고정 초모수(최종 M1 Elastic-net):", bp)
spec_enet = tmc.model_zoo(tmc.Config(models=["enet"]), M1, [])["enet"]
sizes = [1000, 2000, 3000, 4000, 6000, 8000] if not FAST else [1000, 4000, 8000]; R = 3 if not FAST else 1
rows = []
for n in sizes:
    for r in range(R):
        idx = np.arange(len(dev)) if n >= len(dev) else train_test_split(np.arange(len(dev)), train_size=n, stratify=y, random_state=SEED + r)[0]
        if n >= len(dev) and r > 0: continue
        pipe = tmc.build_pipeline(spec_enet, M1, []); pipe.set_params(**bp)
        p = cross_val_predict(pipe, dev.loc[idx, M1], y[idx], cv=StratifiedKFold(5, shuffle=True, random_state=SEED + r), method="predict_proba")[:, 1]
        rows.append({"n": n, "반복": r, "AUROC": roc_auc_score(y[idx], p), "AUPRC": average_precision_score(y[idx], p)})
LC = pd.DataFrame(rows); LCm = LC.groupby("n").agg(AUROC=("AUROC", "mean"), AUROC_sd=("AUROC", "std"), AUPRC=("AUPRC", "mean"), AUPRC_sd=("AUPRC", "std")).reset_index()
LCm.to_csv(TAB / f"{PFX}table7c_learning_curve.csv", index=False, encoding="utf-8-sig"); print(LCm.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(5.6, 4))
ax.errorbar(LCm.n, LCm.AUROC, yerr=LCm.AUROC_sd.fillna(0), fmt="o-", color=COL["blue"], capsize=3, lw=1.8, ms=6, label="AUROC (5분할 CV, 반복 평균)")
ax.set_xlabel("학습 표본 수"); ax.set_ylabel("AUROC"); ax.set_title("M1 Elastic-net 학습 곡선(개발군)"); ax.set_ylim(0.5, 0.7); ax.grid(color="#e6e6e6", lw=0.6); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(FIG / f"{PFX}fig7b_learning_curve.png", dpi=150); plt.close(); print("그림 저장:", f"{PFX}fig7b_learning_curve.png")

고정 초모수(최종 M1 Elastic-net): {'est__C': np.float64(0.0011371468492586037), 'est__l1_ratio': np.float64(0.046448898171531194)}
   n  AUROC  AUROC_sd  AUPRC  AUPRC_sd
1000 0.5553    0.0198 0.1777    0.0078
2000 0.6129    0.0037 0.2225    0.0129
3000 0.6087    0.0147 0.2266    0.0094
4000 0.6226    0.0134 0.2292    0.0051
6000 0.6237    0.0034 0.2337    0.0035
8000 0.6272       NaN 0.2400       NaN
그림 저장: fig7b_learning_curve.png


## H7. 결과 정의 민감도 (보조)

In [8]:
if RUN_H7:
    rows = []
    for tgt, lab in [("atrophy_pos", "해마 z ≤ −1 (주 정의)"), ("atrophy_pos_b20", "해마 z 하위 20%"), ("composite_pos", "해마 또는 전체 뇌 위축(복합)")]:
        if tgt == "atrophy_pos": d7 = M / f"{prefix}M1"
        else:
            d7 = M / f"method_{PFX}H7_{tgt}_M1"; t0 = time.time()
            tmc.run(dev, "T_ID", tgt, M1, [], None, make_cfg("classification", ["enet", "cat"]), str(d7), resume=True, compute_only=["enet", "cat"]); print(f"{lab}: {time.time() - t0:.0f}초")
        s = pd.read_csv(d7 / "results_summary.csv"); s = s[s.model.isin(["enet", "cat"])].sort_values("AUPRC_mean", ascending=False)
        yy = dev[tgt].values.astype(int)
        for _, r in s.iterrows():
            rows.append({"결과 정의": lab, "양성률": yy.mean(), "모델": r.label, "AUPRC": r.AUPRC_mean, "AUROC": r.AUROC_mean, "순위": int((s.model == r.model).to_numpy().nonzero()[0][0]) + 1})
    H7 = pd.DataFrame(rows); H7.to_csv(TAB / f"{PFX}table7d_outcome_sensitivity.csv", index=False, encoding="utf-8-sig")
    print(H7.round(4).to_string(index=False))
else:
    print("RUN_H7 = False: 건너뜀")

[설정] task=classification  n=8000  p=17  models=['enet', 'cat']  outer=5x2  inner=3  n_iter=40
  Elastic-net (기준선)      repeat 0 fold 0  done (     4s)
  Elastic-net (기준선)      repeat 0 fold 1  done (     6s)
  Elastic-net (기준선)      repeat 0 fold 2  done (     9s)
  Elastic-net (기준선)      repeat 0 fold 3  done (    11s)
  Elastic-net (기준선)      repeat 0 fold 4  done (    16s)
  Elastic-net (기준선)      repeat 1 fold 0  done (    19s)
  Elastic-net (기준선)      repeat 1 fold 1  done (    23s)
  Elastic-net (기준선)      repeat 1 fold 2  done (    25s)
  Elastic-net (기준선)      repeat 1 fold 3  done (    29s)
  Elastic-net (기준선)      repeat 1 fold 4  done (    32s)
  CatBoost               repeat 0 fold 0  done (    12s)
  CatBoost               repeat 0 fold 1  done (    23s)
  CatBoost               repeat 0 fold 2  done (    35s)
  CatBoost               repeat 0 fold 3  done (    46s)
  CatBoost               repeat 0 fold 4  done (    57s)
  CatBoost               repeat 1 fold 0  done (   

## 요약: 표 7, 그림 7, 판정

In [9]:
T7 = pd.DataFrame(RESULTS)
prim = T7[T7["주 비교"]].copy()
# Holm: 주 비교 5건(실행된 것만)
ps = prim["ΔAUPRC p"].to_numpy(); order = np.argsort(ps); adj = np.empty_like(ps); run = 0.0
for rank, i in enumerate(order): run = max(run, min(1.0, (len(ps) - rank) * ps[i])); adj[i] = run
T7["Holm p(주 비교)"] = np.nan; T7.loc[prim.index, "Holm p(주 비교)"] = adj
T7["평가군 2차 사용 조건 충족"] = T7["주 비교"] & (T7["ΔAUPRC"] >= 0.01) & (T7["Holm p(주 비교)"] < 0.05)
T7.to_csv(TAB / f"{PFX}table7_methodology.csv", index=False, encoding="utf-8-sig")
show = T7.copy()
for k in ["AUROC", "AUPRC"]: show[f"Δ{k} (95% CI)"] = [f"{a:+.4f} ({b:+.4f}, {c:+.4f})" for a, b, c in zip(show[f"Δ{k}"], show[f"Δ{k} 하한"], show[f"Δ{k} 상한"])]
pd.set_option("display.width", 250)
print("표 7. 기준 M1 Elastic-net(AUROC %.4f, AUPRC %.4f) 대비" % (roc_auc_score(y, BASE), average_precision_score(y, BASE)))
print(show[["가설", "변형", "주 비교", "AUROC", "AUPRC", "ΔAUPRC (95% CI)", "ΔAUROC (95% CI)", "ΔAUPRC p", "Holm p(주 비교)"]].to_string(index=False, float_format=lambda v: f"{v:.4f}"))
ok = T7[T7["평가군 2차 사용 조건 충족"]]
if len(ok):
    simp = {"H1": 1, "H5": 1, "H2": 2, "H3": 3, "H4": 4}; pick = ok.assign(_s=ok.가설.map(simp)).sort_values(["_s", "ΔAUPRC"], ascending=[True, False]).iloc[0]
    print(f"\n판정: 조건(ΔAUPRC ≥ 0.01, Holm p < 0.05)을 충족하는 주 비교 {len(ok)}건. 사전 규칙에 따라 가장 단순한 '{pick.변형}'({pick.가설}) 하나만 평가군 2차 확인 검정 후보로 기록한다. 이 노트북은 평가군을 읽지 않으며, 2차 사용은 별도 단계에서 변경 기록과 함께 수행한다.")
else:
    print("\n판정: 조건(ΔAUPRC ≥ 0.01, Holm p < 0.05)을 충족하는 가설이 없다. 평가군은 다시 쓰지 않는다. 결과는 '개발군 사후 탐색'으로만 보고한다.")

# 그림 7: ΔAUPRC 숲 그림
fig, ax = plt.subplots(figsize=(7.2, 0.5 * len(T7) + 1.6))
yv = np.arange(len(T7))[::-1]
for i, (_, r) in enumerate(T7.iterrows()):
    c = COL["blue"] if r["주 비교"] else COL["gray"]
    ax.errorbar(r["ΔAUPRC"], yv[i], xerr=[[r["ΔAUPRC"] - r["ΔAUPRC 하한"]], [r["ΔAUPRC 상한"] - r["ΔAUPRC"]]], fmt="o", color=c, capsize=3, ms=6, lw=1.6)
ax.axvline(0, color=COL["ink2"], lw=1); ax.axvline(0.01, color=COL["violet"], lw=1, ls=":"); ax.text(0.0102, yv[0] + 0.5, "판정 기준 +0.01", color=COL["violet"], fontsize=8, va="bottom")
ax.set_yticks(yv); ax.set_yticklabels([f"{h} {v}" for h, v in zip(T7.가설, T7.변형)], fontsize=8.5)
ax.set_xlabel("ΔAUPRC 대 M1 Elastic-net (개발군 OOF, 95% 부트스트랩 CI)"); ax.set_title("방법론 연구: 사후 탐색 결과(개발군 전용)")
ax.plot([], [], "o", color=COL["blue"], label="주 비교(Holm)"); ax.plot([], [], "o", color=COL["gray"], label="보조"); ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(axis="x", color="#e6e6e6", lw=0.6); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(FIG / f"{PFX}fig7_methodology.png", dpi=150, bbox_inches="tight"); plt.close(); print("그림 저장:", f"{PFX}fig7_methodology.png")

표 7. 기준 M1 Elastic-net(AUROC 0.6306, AUPRC 0.2394) 대비
가설                           변형  주 비교  AUROC  AUPRC            ΔAUPRC (95% CI)            ΔAUROC (95% CI)  ΔAUPRC p  Holm p(주 비교)
H1          Elastic-net 회귀 → 순위  True 0.6268 0.2382 -0.0011 (-0.0083, +0.0061) -0.0039 (-0.0143, +0.0067)    0.7845        1.0000
H1             CatBoost 회귀 → 순위 False 0.6233 0.2340 -0.0054 (-0.0129, +0.0015) -0.0073 (-0.0196, +0.0049)    0.1235           NaN
H2              M1+ Elastic-net  True 0.6248 0.2374 -0.0020 (-0.0081, +0.0040) -0.0058 (-0.0108, -0.0011)    0.5005        1.0000
H2                 M1+ CatBoost False 0.6279 0.2369 -0.0024 (-0.0107, +0.0063) -0.0028 (-0.0104, +0.0053)    0.5545           NaN
H3          로지스틱 스태킹(교차검증 메타학습)  True 0.6274 0.2376 -0.0018 (-0.0044, +0.0008) -0.0032 (-0.0059, -0.0004)    0.1775        0.7100
H3                     로짓 단순 평균 False 0.6276 0.2345 -0.0049 (-0.0117, +0.0027) -0.0030 (-0.0107, +0.0042)    0.1935           NaN
H4            TabPFN-3.5 앙상블 32  Tru